In [1]:
#imports and methods

import pandas as pd
import numpy as np
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, hamming_loss

def compute_metrics(labels, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    hl = hamming_loss(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'hamming_loss': hl
    }

def print_metrics(name, metrics):
    print("\n======================")
    print(name)
    print("======================")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")


In [2]:
# cargar datos

train = pd.read_csv("dataset_train.csv")
test = pd.read_csv("dataset_test.csv")

print(train.head())

           movie_name                            genre  \
0         Silent Hill                  Horror, Mystery   
1  Breaking the Waves                   Drama, Romance   
2          Wind Chill          Drama, Horror, Thriller   
3         Godmothered          Family, Fantasy, Comedy   
4         Donkey Skin  Fantasy, Comedy, Music, Romance   

                                         description  
0  Rose, a desperate mother takes her adopted dau...  
1  In a small and conservative Scottish village, ...  
2  Two college students share a ride home for the...  
3  A young and unskilled fairy godmother that ven...  
4  A fairy godmother helps a princess disguise he...  


In [3]:
# preprocess y binarización

train["genre"] = train["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(train["genre"])

print(mlb.classes_)
print(len(mlb.classes_))

# split
X = train["description"].fillna("")
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# TF-IDF
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    stop_words='english',
    lowercase=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(test['description'])

['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Drama' 'Family'
 'Fantasy' 'History' 'Horror' 'Music' 'Mystery' 'Romance'
 'Science Fiction' 'TV Movie' 'Thriller' 'War' 'Western']
18


In [4]:
#modelo 1 - logistic regression

modelo1 = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
modelo1.fit(X_train_tfidf, y_train)

y_proba = modelo1.predict_proba(X_val_tfidf)

threshold = 0.5
y_pred = (y_proba >= threshold).astype(int)

metrics = compute_metrics(y_val, y_pred)
print_metrics('Modelo 1', metrics)



Modelo 1
accuracy: 0.1156
f1: 0.5346
precision: 0.5108
recall: 0.5639
hamming_loss: 0.1190


In [5]:
#modelo 2 - svm

tfidf_svm = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9,
    stop_words='english'
)

X_train_svm = tfidf_svm.fit_transform(X_train)
X_val_svm = tfidf_svm.transform(X_val)

X_test_svm = tfidf_svm.transform(test["description"])

modelo2 = OneVsRestClassifier(
    LinearSVC(C=1.5, class_weight='balanced')
)

modelo2.fit(X_train_svm, y_train)

y_pred = modelo2.predict(X_val_svm)

metrics = compute_metrics(y_val, y_pred)
print_metrics('Modelo 2', metrics)


Modelo 2
accuracy: 0.1285
f1: 0.5018
precision: 0.5729
recall: 0.4580
hamming_loss: 0.1174


In [7]:
#modelo 3 - TF-IDF, regresión Logística One-vs-Rest y selección Top-K

# cargar datos

train = pd.read_csv("dataset_train.csv")
test = pd.read_csv("dataset_test.csv")

train["description"] = train["description"].fillna("")
test["description"] = test["description"].fillna("")

train["genre"] = train["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(train["genre"])

X = train["description"]

# split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

# features

features = FeatureUnion([
    ("word", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        max_features=50000,
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        stop_words="english"
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=30000,
        min_df=2,
        sublinear_tf=True
    ))
])

X_train_vec = features.fit_transform(X_train)
X_val_vec = features.transform(X_val)

# modelo

modelo3 = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        C=2.0
    )
)

modelo3.fit(X_train_vec, y_train)

proba_val = modelo3.predict_proba(X_val_vec)

# función TOP-K

def predict_with_topk(proba, threshold=0.5, max_labels=3):
    preds = np.zeros_like(proba, dtype=int)

    for i in range(proba.shape[0]):
        selected = np.where(proba[i] >= threshold)[0]

        if len(selected) == 0:
            selected = [np.argmax(proba[i])]

        if len(selected) > max_labels:
            selected = selected[np.argsort(proba[i][selected])[-max_labels:]]

        preds[i, selected] = 1

    return preds

# buscar mejor compromiso

best_score = -1
best_params = None
best_metrics = None

for threshold in np.arange(0.50, 0.75, 0.01):
    for max_labels in [2, 3, 4]:
        preds = predict_with_topk(
            proba_val,
            threshold=threshold,
            max_labels=max_labels
        )

        metrics = compute_metrics(y_val, preds)

        score = metrics["f1"] + 0.5 * metrics["accuracy"]

        if score > best_score:
            best_score = score
            best_params = (threshold, max_labels)
            best_metrics = metrics

print("Best params:", best_params)
print_metrics("Modelo 3", best_metrics)

# reentrenar con todo

X_all_vec = features.fit_transform(X)

modelo_final = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        C=2.0
    )
)

modelo_final.fit(X_all_vec, y)

proba_test = modelo_final.predict_proba(features.transform(test["description"]))

# predicción final

best_threshold, best_max_labels = best_params

preds_test = predict_with_topk(
    proba_test,
    threshold=best_threshold,
    max_labels=best_max_labels
)

genres_pred = mlb.inverse_transform(preds_test)

test["genre"] = [", ".join(g) for g in genres_pred]

# guardar CSV + ZIP

submission = test[["movie_name", "description", "genre"]]

submission.to_csv("dataset_test_preds.csv", index=False)

with zipfile.ZipFile("submission_modelo3.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("dataset_test_preds.csv")

print("\nArchivo creado: submission_modelo3.zip")
print(submission.head(10))

Best params: (np.float64(0.5), 3)

Modelo 3
accuracy: 0.1474
f1: 0.5333
precision: 0.5880
recall: 0.4954
hamming_loss: 0.1093

Archivo creado: submission_modelo3.zip
                            movie_name  \
0                    Opposites Attract   
1  A Turtle's Tale: Sammy's Adventures   
2            My Stepmother Is an Alien   
3                      You've Got Mail   
4                            The Thing   
5                            Malicious   
6                                 Howl   
7                               Spring   
8                             The Trip   
9                 Secret in Their Eyes   

                                         description  \
0  She's a divorce lawyer, single mother and perp...   
1  A sea turtle who was hatched in 1959 spends th...   
2  Trying to rescue her home planet from destruct...   
3  Book superstore magnate, Joe Fox and independe...   
4  In the winter of 1982, a twelve-man research t...   
5  A widower and two of his sons be

In [12]:
#modelo 4 - transformers RoBERTa

from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer)

# cargar datos

train = pd.read_csv("dataset_train.csv")
test = pd.read_csv("dataset_test.csv")

train["description"] = train["description"].fillna("")
test["description"] = test["description"].fillna("")

train["genre"] = train["genre"].apply(
    lambda x: [g.strip() for g in x.split(",")]
)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(train["genre"])

X = train["description"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42
)

# datasets

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df = pd.DataFrame({
    "description": X_train.values,
    "labels": list(y_train.astype(float))
})

val_df = pd.DataFrame({
    "description": X_val.values,
    "labels": list(y_val.astype(float))
})

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

def tokenize_function(batch):
    return tokenizer(
        batch["description"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# modelo 4 (RoBERTa)

modelo4 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

# entrenamiento

training_args = TrainingArguments(
    output_dir="./results_modelo4",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none"
)

trainer_modelo4 = Trainer(
    model=modelo4,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer_modelo4.train()

# validación

val_predictions = trainer_modelo4.predict(val_dataset)

val_logits = val_predictions.predictions
val_labels = val_predictions.label_ids

val_probs = 1 / (1 + np.exp(-val_logits))

# top-K

def predict_with_topk(proba, threshold=0.35, max_labels=3):
    preds = np.zeros_like(proba, dtype=int)

    for i in range(proba.shape[0]):
        selected = np.where(proba[i] >= threshold)[0]

        if len(selected) == 0:
            selected = [np.argmax(proba[i])]

        if len(selected) > max_labels:
            selected = selected[np.argsort(proba[i][selected])[-max_labels:]]

        preds[i, selected] = 1

    return preds

# buscar mejor parámetros

best_score = -1
best_params = None
best_metrics = None

for threshold in np.arange(0.20, 0.55, 0.01):
    for max_labels in [2, 3, 4]:
        preds_val = predict_with_topk(val_probs, threshold, max_labels)

        metrics = compute_metrics(val_labels, preds_val)

        score = metrics["f1"] + 0.5 * metrics["accuracy"]

        if score > best_score:
            best_score = score
            best_params = (threshold, max_labels)
            best_metrics = metrics

print("Best params:", best_params)
print_metrics("Modelo 4 - RoBERTa", best_metrics)

# predicción final y CSV

test_df = pd.DataFrame({
    "description": test["description"].values
})

test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_function, batched=True)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

test_predictions = trainer_modelo4.predict(test_dataset)

test_logits = test_predictions.predictions
test_probs = 1 / (1 + np.exp(-test_logits))

threshold, max_labels = best_params

preds_test = predict_with_topk(test_probs, threshold, max_labels)

genres_pred = mlb.inverse_transform(preds_test)

test["genre"] = [", ".join(g) for g in genres_pred]

submission = test[["movie_name", "description", "genre"]]

submission.to_csv("dataset_test_preds.csv", index=False)

with zipfile.ZipFile("submission_modelo4.zip", "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("dataset_test_preds.csv")

print("\nArchivo creado: submission_modelo4.zip")
print(submission.head(10))


Map:   0%|          | 0/7627 [00:00<?, ? examples/s]

Map:   0%|          | 0/848 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,0.262215,0.250162
2,0.215096,0.228324
3,0.193830,0.224828


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Best params: (np.float64(0.3000000000000001), 3)

Modelo 4 - RoBERTa
accuracy: 0.1958
f1: 0.5616
precision: 0.5983
recall: 0.5424
hamming_loss: 0.0956


Map:   0%|          | 0/942 [00:00<?, ? examples/s]


Archivo creado: submission_modelo4.zip
                            movie_name  \
0                    Opposites Attract   
1  A Turtle's Tale: Sammy's Adventures   
2            My Stepmother Is an Alien   
3                      You've Got Mail   
4                            The Thing   
5                            Malicious   
6                                 Howl   
7                               Spring   
8                             The Trip   
9                 Secret in Their Eyes   

                                         description  \
0  She's a divorce lawyer, single mother and perp...   
1  A sea turtle who was hatched in 1959 spends th...   
2  Trying to rescue her home planet from destruct...   
3  Book superstore magnate, Joe Fox and independe...   
4  In the winter of 1982, a twelve-man research t...   
5  A widower and two of his sons become infatuate...   
6  When passengers on a train are attacked by a c...   
7  A young man in a personal tailspin flees the U